# 1. Distributed

PyTorch for default run in one process, one GPU. So to use multiple GPUs we need multiple process, one per GPU, the `torch.distributed` it's the lib where we can do that

### 1. `torchrun`

Processo que permite rodar o script N vezes ao mesmo tempo (criar N processos), alem de podemos setar mais configurações como:
- Processos por máquina
- Quantas máquinas
- Máquina principal

### 1.1 `.init_process_group(backend='nccl')`

Todas as GPUs existem mas não sabem da existência das outras, 
são isoladas. O `init_process_group` conecta esses processos, 
e o `backend='nccl'` define que a comunicação entre elas vai 
usar o NCCL da NVIDIA, que é otimizado para GPUs.

In [ ]:
import torch.distributed as dist

dist.init_process_group(backend='nccl')

### 1.2 Device Mesh 



Depois do process_group as GPUs já estão conectadas, mas o PyTorch precisa ter uma ordem para isso, então criamos uma ordem

In [ ]:
import torch.distributed.device_mesh import device_mesh

mesh = init_device_mesh("cuda", (X,))

### 1.3 Tensor Parallel (`parallelize_module`)

Pega o modelo original, nesse caso (1024, 1024) e divide em X GPUs que foram informados no mesh. Temos 2 opções de divisão

- Colunas (`ColwiseParallel`)
- Linhas (`RowwiseParallel`)


Para ver se foi bem sucedido, podemos usar o rank


In [ ]:
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.distributed.device_mesh import init_device_mesh
from torch.distributed.tensor.parallel import parallelize_module, ColwiseParallel

dist.init_process_group(backend="nccl")
mesh = init_device_mesh("cuda", (4,))

model = nn.Linear(1024, 1024).cuda()

model = parallelize_module(
    model,
    mesh,
    ColwiseParallel()
)

rank = dist.get_rank()
print(f"Rank {rank} → Weight shape: {model.weight.shape}")

se quisermos em MLP, usaremos de outra forma o `parallelize_module`, e podemos intercalar c/a estratégia

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(1024, 4096)  # ColwiseParallel
        self.fc2 = nn.Linear(4096, 1024)  # RowwiseParallel
        self.act = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

model = MLP().cuda()

model = parallelize_module(
    model,
    mesh,
    {
        "fc1": ColwiseParallel(),
        "fc2": RowwiseParallel(),
    }
)